<a href="https://colab.research.google.com/github/BenayaL/Cloud_Computing_Project/blob/main/Tirgul1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Load and prepare the data (Using our clean reading method)
file_path = '/content/drive/My Drive/students.txt'
column_headers = ['First Name', 'Last Name', 'Email', 'Courses Taken', 'Interesting Link']

try:
    df = pd.read_csv(file_path, sep=r'\s*\|\s*', header=None, engine='python')
    df = df.iloc[:, :5] # Keep only the first 5 columns
    df.columns = column_headers
    df.dropna(how='all', inplace=True)
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}")
    # Creating dummy data if file is missing
    df = pd.DataFrame([["Uriel", "Pekelis", "uriel@email.com", "Cloud", "link"]], columns=column_headers)

# Create a "Full Name" column to make the dropdown look nice
df['Full Name'] = df['First Name'] + " " + df['Last Name']

# Add a "Favorite Program" column if we haven't already
if 'Favorite Program' not in df.columns:
    df['Favorite Program'] = "Not assigned yet"

# 2. Create the Widgets
# The dropdown uses the 'Full Name' column for its options
student_dropdown = widgets.Dropdown(
    options=['-- Select a Student --'] + df['Full Name'].tolist(),
    description='Student:',
    disabled=False,
)

info_output = widgets.Output() # Area to show the student's static info
status_output = widgets.Output() # Area to show success messages

# Editing widgets (Disabled by default until a student is selected)
program_input = widgets.Text(description='Fav Program:', disabled=True)
save_button = widgets.Button(description='Save Edit', button_style='success', disabled=True)

# 3. Define the interaction functions
def on_dropdown_change(change):
    """This runs every time a new name is picked from the dropdown."""
    if change['type'] == 'change' and change['name'] == 'value':
        selected_name = change['new']

        info_output.clear_output()
        status_output.clear_output()

        # If they go back to the default prompt, disable everything
        if selected_name == '-- Select a Student --':
            program_input.value = ''
            program_input.disabled = True
            save_button.disabled = True
            return

        # Enable the editing tools
        program_input.disabled = False
        save_button.disabled = False

        # Find the specific student's row in the DataFrame
        # .iloc[0] grabs the first matching row as a dictionary-like object
        student_data = df[df['Full Name'] == selected_name].iloc[0]

        # Display their information
        with info_output:
            print(f"--- Information for {selected_name} ---")
            print(f"Email:   {student_data['Email']}")
            print(f"Courses: {student_data['Courses Taken']}")
            print(f"Link:    {student_data['Interesting Link']}")

        # Pre-fill the input box with whatever their current favorite program is
        program_input.value = student_data['Favorite Program']

def on_save_clicked(b):
    """This runs when the Save button is clicked."""
    selected_name = student_dropdown.value
    new_program = program_input.value

    # 1. Update the DataFrame in memory
    row_index = df[df['Full Name'] == selected_name].index[0]
    df.at[row_index, 'Favorite Program'] = new_program

    # 2. Save the updated DataFrame back to the text file
    try:
        # We drop 'Full Name' before saving so we don't change the original file structure
        cols_to_save = [c for c in df.columns if c != 'Full Name']

        df[cols_to_save].to_csv(
            file_path,
            sep='|',
            index=False,
            header=False
        )

        with status_output:
            clear_output()
            print(f"✅ Saved to file! Updated {selected_name}'s program to: '{new_program}'")

    except Exception as e:
        with status_output:
            clear_output()
            print(f"❌ Error saving to file: {e}")

# 4. Connect the functions to the widgets
student_dropdown.observe(on_dropdown_change)
save_button.on_click(on_save_clicked)

# 5. Display everything in a neat layout
# HBox puts things side-by-side, VBox stacks them vertically
edit_row = widgets.HBox([program_input, save_button])
ui_layout = widgets.VBox([student_dropdown, info_output, edit_row, status_output])

display(ui_layout)

To fix the GitHub rendering error caused by `metadata.widgets`, you can run the following code. This code will remove the `widgets` metadata from your current notebook, which often resolves the issue. You should run this **before saving and uploading your notebook to GitHub**.